# 07 - K-Means (clustering)

**Idée :** on résume chaque (utilisateur, heure) par quelques chiffres (features), puis K-Means
regroupe les comportements **similaires** en clusters. Un point **loin** des clusters
« normaux » est une **anomalie**.

**Score d'anomalie** = distance au centre du cluster normal le plus proche.

⚠️ Piège : des attaquants très différents peuvent former leur *propre petit cluster* et
paraître « proches de leur centre ». On ne mesure donc la distance qu'aux **gros clusters**
(≥ 5 % des points = comportement normal).

**Entrée :** `datasets/features.csv` (créé dans le notebook 04).
Sans ce fichier, le notebook utilise de **fausses données** (mode démo).

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

sys.path.append(".")
import utils

os.makedirs("../results/figures", exist_ok=True)
os.makedirs("../results/metrics", exist_ok=True)

## 1. Chargement des features

In [ ]:
FEAT_PATH = "../datasets/features.csv"

if os.path.exists(FEAT_PATH):
    feat = pd.read_csv(FEAT_PATH)
    print("Vraies features chargées :", len(feat), "lignes")
else:
    feat = utils.build_features(utils.make_fake_logs())
    print("MODE DEMO (fausses données) :", len(feat), "lignes")

print("Attaques (label=1) :", feat["label"].sum())
feat.head()

## 2. Séparation entraînement / test
K-Means apprend la « normalité » sur la période **avant** la première attaque.

In [ ]:
if feat["label"].sum() > 0:
    TRAIN_END = int(feat.loc[feat["label"] == 1, "hour_id"].min())
else:
    TRAIN_END = int(feat["hour_id"].quantile(0.7))
print("Entraînement : heures <", TRAIN_END, "| Test : heures >=", TRAIN_END)

train = feat[feat["hour_id"] < TRAIN_END]
test = feat[feat["hour_id"] >= TRAIN_END].copy()

## 3. Préparation des features
- `log1p` sur les comptages (ils ont de très grandes valeurs rares),
- `StandardScaler` : toutes les features sur la même échelle (sinon K-Means ne regarde que la plus grande).

In [ ]:
cols = ["n_conn", "n_machines", "n_fail", "fail_ratio"]

def prep(d):
    X = d[cols].copy()
    for c in ["n_conn", "n_machines", "n_fail"]:
        X[c] = np.log1p(X[c])
    return X

scaler = StandardScaler().fit(prep(train))        # on apprend l'échelle sur le train seulement
X_train = scaler.transform(prep(train))
X_test = scaler.transform(prep(test))

## 4. Choisir k (méthode du coude)
On cherche le k où la courbe arrête de baisser vite.

In [ ]:
ks = range(1, 9)
inertias = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_train).inertia_ for k in ks]

plt.figure(figsize=(5, 3.5))
plt.plot(list(ks), inertias, "o-")
plt.xlabel("k (nombre de clusters)"); plt.ylabel("inertie"); plt.title("Méthode du coude")
plt.tight_layout()
plt.savefig("../results/figures/kmeans_elbow.png", dpi=150)
plt.show()

## 5. Entraînement et score d'anomalie

In [ ]:
K = 3                                            # à ajuster avec le coude
km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(X_train)

sizes = np.bincount(km.labels_, minlength=K) / len(X_train)
big = np.where(sizes >= 0.05)[0]                 # gros clusters = comportement normal
print("Taille des clusters :", np.round(sizes, 3), "| gros clusters :", big)

dist = km.transform(X_test)                      # distance de chaque point à chaque centre
test["score_kmeans"] = dist[:, big].min(axis=1)  # distance au gros cluster le plus proche
test.nlargest(10, "score_kmeans")[["user", "hour_id"] + cols + ["label", "score_kmeans"]]

## 6. Évaluation
PR-AUC (à comparer à la proportion d'attaques = niveau du hasard), ROC-AUC, précision sur le top 20.

In [ ]:
y, s = test["label"], test["score_kmeans"]
metrics = {"modele": "kmeans", "k": K, "n_test": len(test), "n_attaques": int(y.sum()),
           "pr_auc_hasard": round(y.mean(), 4)}

if y.sum() > 0:
    metrics["pr_auc"] = round(average_precision_score(y, s), 4)
    metrics["roc_auc"] = round(roc_auc_score(y, s), 4)
    metrics["precision@20"] = round(test.nlargest(20, "score_kmeans")["label"].mean(), 4)

pd.DataFrame([metrics]).to_csv("../results/metrics/kmeans_metrics.csv", index=False)
test[["user", "hour_id", "label", "score_kmeans"]].to_csv("../results/metrics/kmeans_scores.csv", index=False)
metrics

## 7. Graphiques

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

# (a) clusters en 2D (PCA) : on projette les features sur 2 axes pour les voir
pca = PCA(n_components=2).fit(X_train)
P = pca.transform(X_test)
ax[0].scatter(P[y == 0, 0], P[y == 0, 1], s=8, alpha=.4, label="normal", color="tab:blue")
if y.sum() > 0:
    ax[0].scatter(P[y == 1, 0], P[y == 1, 1], s=25, label="attaque", color="tab:red")
C = pca.transform(km.cluster_centers_)
ax[0].scatter(C[:, 0], C[:, 1], marker="X", s=150, color="black", label="centres")
ax[0].set_title("Données de test (PCA 2D)"); ax[0].legend()

# (b) distribution des scores
ax[1].hist(test.loc[y == 0, "score_kmeans"], bins=40, alpha=.7, label="normal", color="tab:blue")
if y.sum() > 0:
    ax[1].hist(test.loc[y == 1, "score_kmeans"], bins=40, alpha=.8, label="attaque", color="tab:red")
ax[1].set_yscale("log"); ax[1].set_xlabel("distance au cluster normal"); ax[1].legend()
ax[1].set_title("Distribution des scores")

# (c) précision-rappel
if y.sum() > 0:
    p, r, _ = precision_recall_curve(y, s)
    ax[2].plot(r, p, color="tab:green")
    ax[2].axhline(y.mean(), ls="--", color="gray", label="hasard")
    ax[2].set_xlabel("rappel"); ax[2].set_ylabel("précision"); ax[2].legend()
    ax[2].set_title(f"Précision-rappel (PR-AUC = {metrics['pr_auc']})")

plt.tight_layout()
plt.savefig("../results/figures/kmeans_overview.png", dpi=150)
plt.show()

## 8. Top 20 des utilisateurs les plus suspects

In [ ]:
top_users = test.groupby("user")["score_kmeans"].max().sort_values(ascending=False).head(20)
plt.figure(figsize=(8, 5))
top_users[::-1].plot.barh(color="tab:green")
plt.xlabel("score K-Means maximal"); plt.title("Top 20 utilisateurs suspects (K-Means)")
plt.tight_layout()
plt.savefig("../results/figures/kmeans_top20_users.png", dpi=150)
plt.show()

## 9. À retenir / limites
- Simple et rapide, mais dépend du choix de **k** et des **features**.
- Ne voit pas l'**ordre** des événements (c'est le rôle de la chaîne de Markov).
- Résultats à comparer avec les autres modèles dans le notebook 08.